# 🧠 Neural–Symbolic Knowledge Base Construction System

## A Publication-Level End-to-End Generative AI System

---

### Overview

This notebook implements a complete **Neural–Symbolic Knowledge Base Construction System** that transforms raw unstructured text into a structured, queryable knowledge graph. The system blends the power of large language models (LLMs) for knowledge extraction with symbolic reasoning rules to enforce logical consistency, producing a reliable and explainable knowledge representation.

The pipeline is organized into modular phases:

```
Raw Text → Preprocessing → LLM Extraction → Triple Parsing → Confidence Scoring
        → Symbolic Reasoning → Knowledge Graph → Storage → Inference → Query → Evaluation
```

Each module is independently testable and callable via backend API functions that the Streamlit frontend uses directly.

**Prerequisites:**
- Ollama running locally with LLaMA3 pulled (`ollama pull llama3`)
- All Python dependencies installed (`pip install -r requirements.txt`)
- Run the Streamlit frontend with: `streamlit run app.py`

---
## Phase 1 — System Initialization

We begin by verifying the environment, importing all required libraries, and confirming that the local Ollama LLM service is reachable. If Ollama is unavailable, the system falls back to a deterministic rule-based extractor so the rest of the pipeline still runs.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Phase 1.1 – Library imports and environment check
# ─────────────────────────────────────────────────────────────
import re
import json
import time
import uuid
import copy
import hashlib
import logging
import warnings
import itertools
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any
from collections import defaultdict, deque
from datetime import datetime

import requests
import networkx as nx
import numpy as np

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger(__name__)

print('✅ Core libraries loaded successfully.')

In [ ]:
# ─────────────────────────────────────────────────────────────
# Phase 1.2 – Ollama connectivity check and model validation
# ─────────────────────────────────────────────────────────────
OLLAMA_BASE_URL = 'http://localhost:11434'
OLLAMA_MODEL    = 'llama3'          # Change to 'llama3:latest' if needed
OLLAMA_TIMEOUT  = 120               # seconds
OLLAMA_AVAILABLE = False

def _check_ollama() -> bool:
    """Return True if Ollama is reachable and the target model is available."""
    try:
        resp = requests.get(f'{OLLAMA_BASE_URL}/api/tags', timeout=5)
        if resp.status_code != 200:
            return False
        models = [m['name'] for m in resp.json().get('models', [])]
        if any(OLLAMA_MODEL in m for m in models):
            return True
        # Attempt pull (non-blocking)
        logger.info(f"Model '{OLLAMA_MODEL}' not found locally. Attempting pull…")
        requests.post(f'{OLLAMA_BASE_URL}/api/pull',
                      json={'name': OLLAMA_MODEL, 'stream': False},
                      timeout=10)
        return True
    except Exception as exc:
        logger.warning(f'Ollama not reachable: {exc}')
        return False

OLLAMA_AVAILABLE = _check_ollama()
status = '✅ Online' if OLLAMA_AVAILABLE else '⚠️  Offline — rule-based fallback active'
print(f'Ollama status: {status}')
print(f'Target model : {OLLAMA_MODEL}')

---
## Phase 2 — Text Preprocessing

Raw user text passes through a cleaning layer before reaching the LLM. This layer handles noise removal, whitespace normalisation, and sentence boundary detection. Clean, well-segmented sentences improve extraction accuracy considerably.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Phase 2 – Preprocessing Module
# ─────────────────────────────────────────────────────────────

class Preprocessor:
    """
    Cleans and segments raw text before it is passed to the LLM.
    Works without external NLP libraries so it stays lightweight
    and always available.
    """

    # Characters that are purely decorative noise
    _NOISE_RE = re.compile(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]')
    # Collapse multiple blank lines
    _BLANK_RE = re.compile(r'\n{3,}')
    # Sentence boundary: '. ', '! ', '? '
    _SENT_RE  = re.compile(r'(?<=[.!?])\s+')

    @classmethod
    def clean(cls, text: str) -> str:
        """Remove control characters and normalise whitespace."""
        text = cls._NOISE_RE.sub(' ', text)
        text = text.replace('\t', ' ')
        text = cls._BLANK_RE.sub('\n\n', text)
        text = re.sub(r' {2,}', ' ', text)
        return text.strip()

    @classmethod
    def segment(cls, text: str) -> List[str]:
        """Split text into individual sentences."""
        cleaned = cls.clean(text)
        sentences = cls._SENT_RE.split(cleaned)
        return [s.strip() for s in sentences if len(s.strip()) > 5]

    @classmethod
    def process(cls, text: str) -> Dict[str, Any]:
        """Full preprocessing pipeline; returns metadata alongside clean text."""
        original   = text
        cleaned    = cls.clean(text)
        sentences  = cls.segment(cleaned)
        return {
            'original'  : original,
            'cleaned'   : cleaned,
            'sentences' : sentences,
            'sent_count': len(sentences),
        }


# ── Quick smoke-test ──────────────────────────────────────────
sample_output = Preprocessor.process(
    'Albert Einstein was born in Germany. He developed the theory of relativity.'
    '   Marie Curie won two Nobel Prizes!'
)
print(json.dumps(sample_output, indent=2))

---
## Phase 3 — Neural Extraction (LLM via Ollama)

The core extraction engine sends a carefully engineered prompt to the local LLM and parses the response. A retry mechanism handles occasional formatting deviations from the model. When Ollama is unavailable a deterministic fallback extractor activates, so the pipeline remains functional during development.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Phase 3 – Neural Extraction Module
# ─────────────────────────────────────────────────────────────

EXTRACTION_PROMPT_TEMPLATE = """\
You are an expert information-extraction engine.
Extract ALL factual knowledge triples from the text below.

STRICT OUTPUT RULES:
- Each triple must be on its own line in EXACTLY this format: (subject, relation, object)
- Use lowercase for relation labels, e.g. 'born_in', 'works_at', 'developed'
- NO explanations, NO numbering, NO markdown, NO extra text — triples only.

TEXT:
{text}

TRIPLES:"""


class NeuralExtractor:
    """Calls the local Ollama LLM to extract knowledge triples from text."""

    def __init__(self, model: str = OLLAMA_MODEL,
                 base_url: str = OLLAMA_BASE_URL,
                 max_retries: int = 3):
        self.model       = model
        self.base_url    = base_url
        self.max_retries = max_retries
        self._endpoint   = f'{base_url}/api/generate'

    def _call_llm(self, prompt: str) -> str:
        """POST to Ollama and return the raw text response."""
        payload = {
            'model' : self.model,
            'prompt': prompt,
            'stream': False,
            'options': {'temperature': 0.1, 'top_p': 0.9},
        }
        resp = requests.post(self._endpoint, json=payload,
                             timeout=OLLAMA_TIMEOUT)
        resp.raise_for_status()
        return resp.json().get('response', '')

    def extract(self, text: str) -> List[str]:
        """
        Extract raw triple strings from text using the LLM.
        Returns a list of strings like '(subject, relation, object)'.
        """
        prompt = EXTRACTION_PROMPT_TEMPLATE.format(text=text)
        for attempt in range(1, self.max_retries + 1):
            try:
                raw = self._call_llm(prompt)
                lines = [l.strip() for l in raw.split('\n') if l.strip()]
                # Keep only lines that look like triples
                triples = [l for l in lines if re.search(r'\(.+,.+,.+\)', l)]
                if triples:
                    return triples
                logger.warning(f'Attempt {attempt}: no valid triples found, retrying…')
            except Exception as exc:
                logger.warning(f'Attempt {attempt} failed: {exc}')
            time.sleep(1)
        logger.error('All LLM extraction attempts failed.')
        return []


# ─────────────────────────────────────────────────────────────
# Fallback: Rule-Based Extractor (when Ollama is offline)
# ─────────────────────────────────────────────────────────────

class RuleBasedExtractor:
    """
    Lightweight pattern-based extractor used when the LLM is unavailable.
    Handles common English sentence structures like:
      "X is/was a Y", "X born in Y", "X founded Y", etc.
    """

    _PATTERNS = [
        # "X was born in Y"
        (r'([A-Z][\w\s]+?) was born in ([\w\s,]+)',
         lambda m: f'({m.group(1).strip()}, born_in, {m.group(2).strip()})'),
        # "X is a/an Y" or "X is the Y"
        (r'([A-Z][\w\s]+?) is (?:a |an |the )?([\w\s]+)',
         lambda m: f'({m.group(1).strip()}, is_a, {m.group(2).strip()})'),
        # "X was/is the Y of Z"
        (r'([A-Z][\w\s]+?) was (?:a |an |the )?([\w\s]+?) of ([A-Z][\w\s]+)',
         lambda m: f'({m.group(1).strip()}, {m.group(2).strip().replace(" ","_")}, {m.group(3).strip()})'),
        # "X founded/created/developed Y"
        (r'([A-Z][\w-]{1,30}(?:\s+[\w-]+){0,3}) (founded|created|developed|invented|discovered) ([\w\s]+)',
         lambda m: f'({m.group(1).strip()}, {m.group(2).strip()}, {m.group(3).strip()})'),
        # "X works/worked at Y"
        (r'([A-Z][\w\s]+?) (?:works|worked) at ([A-Z][\w\s]+)',
         lambda m: f'({m.group(1).strip()}, works_at, {m.group(2).strip()})'),
        # "X won X prize"
        (r'([A-Z][\w\s]+?) won (?:the )?([\w\s]+? (?:Prize|Award|Medal))',
         lambda m: f'({m.group(1).strip()}, won, {m.group(2).strip()})'),
        # "X located in / situated in Y"
        (r'([A-Z][\w\s]+?) (?:is )?located in ([\w\s,]+)',
         lambda m: f'({m.group(1).strip()}, located_in, {m.group(2).strip()})'),
        # "X studied at Y"
        (r'([A-Z][\w\s]+?) studied at ([A-Z][\w\s]+)',
         lambda m: f'({m.group(1).strip()}, studied_at, {m.group(2).strip()})'),
    ]

    def extract(self, text: str) -> List[str]:
        triples: List[str] = []
        for pattern, formatter in self._PATTERNS:
            for match in re.finditer(pattern, text):
                triples.append(formatter(match))
        return list(dict.fromkeys(triples))  # preserve order, deduplicate


_neural_extractor     = NeuralExtractor()
_rule_based_extractor = RuleBasedExtractor()

print('✅ Neural extraction and fallback extractors initialised.')

---
## Phase 4 — Triplet Parsing

LLM output is rarely perfectly formatted. This module robustly parses the raw output, tolerates minor deviations, and produces clean `(subject, relation, object)` dictionaries that subsequent modules can work with reliably.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Phase 4 – Triplet Parser
# ─────────────────────────────────────────────────────────────

class TripletParser:
    """
    Converts raw LLM or rule-based output strings into structured
    triple dictionaries: {'subject': str, 'relation': str, 'object': str}.
    Uses comma-split on the first two commas so the object field may
    contain commas (e.g. 'Warsaw, Poland').
    """

    # Match content between outermost parens
    _OUTER_RE = re.compile(r'\(([^()]+)\)')

    @classmethod
    def _normalise(cls, token: str) -> str:
        """Strip quotes, brackets, and extra whitespace from a token."""
        token = re.sub(r'["\'\'\"\[\]`]', '', token)
        token = re.sub(r'\s+', ' ', token)
        return token.strip()

    @classmethod
    def _normalise_relation(cls, relation: str) -> str:
        """Convert relation to snake_case."""
        rel = cls._normalise(relation).lower()
        rel = re.sub(r'[^a-z0-9]+', '_', rel).strip('_')
        return rel

    @classmethod
    def parse_line(cls, line: str) -> Optional[Dict[str, str]]:
        """Parse a single raw triple string into a dict."""
        m = cls._OUTER_RE.search(line)
        content = m.group(1) if m else line
        parts = content.split(',', 2)
        if len(parts) != 3:
            return None
        subj = cls._normalise(parts[0])
        rel  = cls._normalise_relation(parts[1])
        obj  = cls._normalise(parts[2])
        if not (subj and rel and obj):
            return None
        return {'subject': subj, 'relation': rel, 'object': obj}

    @classmethod
    def parse_all(cls, raw_lines: List[str]) -> List[Dict[str, str]]:
        """Parse a list of raw output lines; silently drop malformed entries."""
        parsed, seen = [], set()
        for line in raw_lines:
            triple = cls.parse_line(line)
            if triple:
                key = (triple['subject'], triple['relation'], triple['object'])
                if key not in seen:
                    seen.add(key)
                    parsed.append(triple)
        return parsed


# Quick test
test_lines = [
    '(Albert Einstein, born_in, Germany)',
    '  Albert Einstein, developed, Theory of Relativity',
    '(Marie Curie, born_in, Warsaw, Poland)',
    '(, , )',   # should be dropped
]
parsed = TripletParser.parse_all(test_lines)
for t in parsed:
    print(t)


---
## Phase 5 — Confidence Estimation

Each extracted triple receives a confidence score that reflects how trustworthy the extraction is. The estimator combines structural features of the triple (entity length, relation specificity, completeness) with an optional LLM-scored heuristic. A configurable threshold then filters the triple set into accepted and rejected groups, both of which are logged for evaluation.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Phase 5 – Confidence Estimation Module
# ─────────────────────────────────────────────────────────────

CONFIDENCE_THRESHOLD = 0.40  # triples below this score are rejected

# Relations that are considered highly informative
_HIGH_VALUE_RELATIONS = {
    'born_in', 'founded', 'developed', 'invented', 'discovered',
    'won', 'authored', 'works_at', 'studied_at', 'located_in',
    'part_of', 'ceo_of', 'president_of', 'member_of',
}

# Relations that are vague / low-information
_LOW_VALUE_RELATIONS = {
    'is_a', 'has', 'have', 'had', 'be', 'are', 'was', 'were',
    'is', 'said', 'says', 'told',
}


class ConfidenceEstimator:
    """
    Assigns a confidence score in [0, 1] to each triple using a
    combination of structural heuristics.
    """

    def __init__(self, threshold: float = CONFIDENCE_THRESHOLD):
        self.threshold = threshold

    def score(self, triple: Dict[str, str]) -> float:
        """Compute and return a confidence score for one triple."""
        subj = triple['subject']
        rel  = triple['relation']
        obj  = triple['object']

        score = 0.5  # base

        # ── Entity quality ────────────────────────────────────
        # Reward properly capitalised entities (likely named entities)
        if subj and subj[0].isupper():
            score += 0.10
        if obj and obj[0].isupper():
            score += 0.05

        # Penalise overly long entities (likely extraction artefacts)
        if len(subj.split()) > 6:
            score -= 0.10
        if len(obj.split()) > 6:
            score -= 0.10

        # ── Relation quality ─────────────────────────────────
        if rel in _HIGH_VALUE_RELATIONS:
            score += 0.20
        elif rel in _LOW_VALUE_RELATIONS:
            score -= 0.15

        # Penalise very short relations (likely single-word noise)
        if len(rel) < 3:
            score -= 0.15

        # ── Completeness ─────────────────────────────────────
        if len(subj) > 1 and len(obj) > 1:
            score += 0.05

        return round(max(0.0, min(1.0, score)), 3)

    def filter_triples(
        self,
        triples: List[Dict[str, str]],
    ) -> Dict[str, List[Dict[str, Any]]]:
        """
        Score all triples and split them into 'accepted' and 'rejected' groups.
        Returns a dict with both groups plus individual scores.
        """
        accepted, rejected = [], []
        for triple in triples:
            s = self.score(triple)
            record = {**triple, 'confidence': s}
            (accepted if s >= self.threshold else rejected).append(record)
        return {'accepted': accepted, 'rejected': rejected}


_confidence_estimator = ConfidenceEstimator()

# Smoke test
test_triple = {'subject': 'Marie Curie', 'relation': 'won', 'object': 'Nobel Prize'}
print(f"Confidence for test triple: {_confidence_estimator.score(test_triple)}")

---
## Phase 6 — Symbolic Reasoning Engine

The symbolic layer acts as a logical gatekeeper. It applies human-defined rules to the extracted triples: removing triples with banned relations, enforcing entity-type constraints, eliminating duplicates, and normalising entity names. This phase ensures that the knowledge graph contains only coherent, non-redundant information.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Phase 6 – Symbolic Reasoning Engine
# ─────────────────────────────────────────────────────────────

# Relations that are not informative enough to keep
_BANNED_RELATIONS = {
    'is', 'are', 'was', 'were', 'be', 'have', 'has', 'had',
    'said', 'says', 'told', 'think', 'knows',
}

# Canonicalisation map: alias → canonical name
_ENTITY_ALIASES: Dict[str, str] = {
    'einstein'        : 'Albert Einstein',
    'curie'           : 'Marie Curie',
    'marie curie'     : 'Marie Curie',
    'usa'             : 'United States',
    'u.s.'            : 'United States',
    'us'              : 'United States',
    'uk'              : 'United Kingdom',
    'u.k.'            : 'United Kingdom',
}


class SymbolicReasoner:
    """
    Applies symbolic rules to refine the triple set:
    1. Remove banned/vague relations.
    2. Canonicalise entity names.
    3. Deduplicate.
    4. Check basic type constraints.
    """

    def __init__(self,
                 banned_relations: Optional[set] = None,
                 aliases: Optional[Dict[str, str]] = None):
        self.banned_relations = banned_relations or _BANNED_RELATIONS
        self.aliases          = aliases or _ENTITY_ALIASES

    def _canonicalise(self, entity: str) -> str:
        """Resolve aliases to their canonical form."""
        return self.aliases.get(entity.lower(), entity)

    def _is_valid(self, triple: Dict[str, Any]) -> bool:
        """Return True only if the triple passes all rule checks."""
        rel  = triple['relation']
        subj = triple['subject']
        obj  = triple['object']

        # Rule 1: reject banned relations
        if rel in self.banned_relations:
            return False

        # Rule 2: subject and object must differ
        if subj.lower() == obj.lower():
            return False

        # Rule 3: entities cannot be empty or too short
        if len(subj) < 2 or len(obj) < 2:
            return False

        # Rule 4: no pure-number entities
        if subj.isdigit() or obj.isdigit():
            return False

        # Rule 5: reject subjects with more than 5 words (likely extraction noise)
        if len(subj.split()) > 5:
            return False

        # Rule 6: reject pronoun subjects (He, She, It, etc.)
        if subj.lower() in {'he', 'she', 'it', 'they', 'we', 'i', 'you',
                             'him', 'her', 'them', 'his', 'its'}:
            return False

        return True

    def apply(self, triples: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """Apply all symbolic rules; return the cleaned triple list."""
        refined, seen = [], set()
        for t in triples:
            t = dict(t)  # work on a copy
            t['subject'] = self._canonicalise(t['subject'])
            t['object']  = self._canonicalise(t['object'])
            if not self._is_valid(t):
                continue
            key = (t['subject'], t['relation'], t['object'])
            if key in seen:
                continue
            seen.add(key)
            refined.append(t)
        return refined


_symbolic_reasoner = SymbolicReasoner()
print('✅ Symbolic reasoning engine ready.')

---
## Phase 7 — Knowledge Graph Construction

Validated triples are loaded into a directed NetworkX graph where each node is an entity and each directed edge carries the relation label and confidence score as attributes. The graph supports efficient adjacency queries, path finding, and subgraph extraction — all needed by the inference and query modules.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Phase 7 – Knowledge Graph Builder
# ─────────────────────────────────────────────────────────────

class KnowledgeGraphBuilder:
    """
    Maintains the directed knowledge graph.
    Nodes  → entities (subject / object strings)
    Edges  → (subject, object) with 'relation' and 'confidence' attributes

    Uses a MultiDiGraph to allow multiple relations between the same entity pair.
    """

    def __init__(self):
        self.graph: nx.MultiDiGraph = nx.MultiDiGraph()
        self._triple_log: List[Dict[str, Any]] = []

    def add_triples(self, triples: List[Dict[str, Any]]) -> None:
        """Insert a list of triple dicts into the graph."""
        for t in triples:
            subj = t['subject']
            rel  = t['relation']
            obj  = t['object']
            conf = t.get('confidence', 1.0)
            self.graph.add_node(subj)
            self.graph.add_node(obj)
            self.graph.add_edge(subj, obj,
                                relation=rel,
                                confidence=conf,
                                timestamp=datetime.utcnow().isoformat())
            self._triple_log.append(t)

    def clear(self) -> None:
        """Reset the graph."""
        self.graph.clear()
        self._triple_log.clear()

    def get_stats(self) -> Dict[str, int]:
        return {
            'nodes': self.graph.number_of_nodes(),
            'edges': self.graph.number_of_edges(),
            'triples_logged': len(self._triple_log),
        }

    def to_serialisable(self) -> Dict[str, Any]:
        """Convert graph to a JSON-serialisable dict."""
        nodes = list(self.graph.nodes())
        edges = []
        for u, v, data in self.graph.edges(data=True):
            edges.append({'from': u, 'to': v, **data})
        return {'nodes': nodes, 'edges': edges}


_kg_builder = KnowledgeGraphBuilder()
print('✅ Knowledge graph builder ready.')

---
## Phase 8 — Storage Manager

The knowledge graph and all extracted artefacts are persisted to disk in JSON format. This allows sessions to be resumed, graphs to be compared across iterations, and triples to be audited outside the notebook.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Phase 8 – Storage Manager
# ─────────────────────────────────────────────────────────────

STORAGE_DIR = Path('kg_storage')
STORAGE_DIR.mkdir(exist_ok=True)

GRAPH_FILE   = STORAGE_DIR / 'knowledge_graph.json'
TRIPLES_FILE = STORAGE_DIR / 'triples_log.json'


class StorageManager:
    """Handles persistence of the knowledge graph and triple logs."""

    @staticmethod
    def save_graph(builder: KnowledgeGraphBuilder) -> Path:
        data = builder.to_serialisable()
        data['saved_at'] = datetime.utcnow().isoformat()
        with open(GRAPH_FILE, 'w') as fh:
            json.dump(data, fh, indent=2)
        logger.info(f'Graph saved → {GRAPH_FILE}')
        return GRAPH_FILE

    @staticmethod
    def load_graph(builder: KnowledgeGraphBuilder) -> bool:
        """
        Reload previously saved triples into builder.
        Returns True on success, False if no file exists.
        """
        if not GRAPH_FILE.exists():
            logger.warning('No saved graph found.')
            return False
        with open(GRAPH_FILE) as fh:
            data = json.load(fh)
        for edge in data.get('edges', []):
            t = {
                'subject'   : edge['from'],
                'relation'  : edge['relation'],
                'object'    : edge['to'],
                'confidence': edge.get('confidence', 1.0),
            }
            builder.add_triples([t])
        logger.info(f'Graph loaded from {GRAPH_FILE}: {builder.get_stats()}')
        return True

    @staticmethod
    def save_triples(triples: List[Dict[str, Any]], tag: str = 'run') -> Path:
        log_file = STORAGE_DIR / f'triples_{tag}_{datetime.utcnow().strftime("%Y%m%d_%H%M%S")}.json'
        with open(log_file, 'w') as fh:
            json.dump(triples, fh, indent=2)
        return log_file


_storage_manager = StorageManager()
print(f'✅ Storage manager ready. Storage dir: {STORAGE_DIR.resolve()}')

---
## Phase 9 — Inference Engine (Multi-hop Reasoning)

The inference engine traverses the graph to discover implicit knowledge that was never directly stated in the original text. Using transitivity rules and configurable depth, it performs multi-hop reasoning and adds inferred triples back into the graph — expanding the knowledge base beyond what was literally extracted.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Phase 9 – Inference Engine
# ─────────────────────────────────────────────────────────────

# Rules as (relation_A, relation_B) → inferred_relation
_INFERENCE_RULES: List[Tuple[str, str, str]] = [
    # Geographic transitivity: born_in city, city located_in country → born_in country
    ('born_in', 'located_in',   'born_in_country'),
    # Employment chain: works_at org, org part_of corporation → employed_by
    ('works_at', 'part_of',     'employed_by'),
    # Academic chain: studied_at university, university located_in city → studied_in
    ('studied_at', 'located_in','studied_in'),
    # Won award, award given_by org → recognised_by
    ('won', 'given_by',         'recognised_by'),
    # General transitivity shortcut (same relation)
    ('located_in', 'located_in','located_in'),
    ('part_of', 'part_of',      'part_of'),
]


class InferenceEngine:
    """
    Applies rule-based multi-hop inference over the knowledge graph,
    discovering and adding implicit triples.
    """

    def __init__(self, max_hops: int = 3):
        self.max_hops = max_hops
        self.inferred: List[Dict[str, Any]] = []

    def _edge_relations(self, graph: nx.MultiDiGraph,
                        u: str, v: str) -> List[str]:
        """Return all relation labels on the edge(s) between u and v."""
        return [data['relation']
                for _, _, data in graph.edges(u, data=True)
                if _ == u and data.get('relation')]

    def run(self, builder: KnowledgeGraphBuilder) -> List[Dict[str, Any]]:
        """
        Perform inference and return newly inferred triples.
        Inferred triples are also added to the graph (confidence = 0.70).
        """
        graph = builder.graph
        new_triples: List[Dict[str, Any]] = []
        seen_keys = set()

        # Build an edge-relation lookup for quick access
        edge_map: Dict[Tuple[str, str], List[str]] = defaultdict(list)
        for u, v, data in graph.edges(data=True):
            edge_map[(u, v)].append(data.get('relation', ''))

        for (rel_a, rel_b, inferred_rel) in _INFERENCE_RULES:
            # Find all edges with rel_a
            for (a, b), rels_ab in edge_map.items():
                if rel_a not in rels_ab:
                    continue
                # Find all edges from b with rel_b
                for (b2, c), rels_bc in edge_map.items():
                    if b2 != b:
                        continue
                    if rel_b not in rels_bc:
                        continue
                    if a == c:
                        continue
                    key = (a, inferred_rel, c)
                    if key in seen_keys:
                        continue
                    seen_keys.add(key)
                    triple = {
                        'subject'   : a,
                        'relation'  : inferred_rel,
                        'object'    : c,
                        'confidence': 0.70,
                        'inferred'  : True,
                        'via'       : f'{a}→[{rel_a}]→{b}→[{rel_b}]→{c}',
                    }
                    new_triples.append(triple)

        if new_triples:
            builder.add_triples(new_triples)
        self.inferred = new_triples
        return new_triples


_inference_engine = InferenceEngine(max_hops=3)
print('✅ Inference engine ready.')

---
## Phase 10 — Query Module

Users can query the knowledge graph in three ways: subject-based ("what do we know about X?"), object-based ("what things relate to Y?"), and relation-based ("who/what are connected by relation Z?"). The query engine translates these natural-language-style questions into graph traversal operations and returns human-readable answers.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Phase 10 – Query Engine
# ─────────────────────────────────────────────────────────────

class QueryEngine:
    """
    Supports three query modes against the knowledge graph:
    - subject  : all facts about a given entity
    - object   : all entities that relate to a given object
    - relation : all triples sharing a specific relation
    
    Also supports free-text queries that are heuristically dispatched.
    """

    def __init__(self, builder: KnowledgeGraphBuilder):
        self.builder = builder

    def _graph(self) -> nx.MultiDiGraph:
        return self.builder.graph

    # ── Structured query methods ──────────────────────────────

    def by_subject(self, subject: str) -> List[Dict[str, Any]]:
        """Return all outgoing edges (facts) for the given subject."""
        results = []
        g = self._graph()
        # Case-insensitive match
        for node in g.nodes():
            if node.lower() == subject.lower():
                for _, obj, data in g.edges(node, data=True):
                    results.append({'subject': node, **data, 'object': obj})
        return results

    def by_object(self, obj: str) -> List[Dict[str, Any]]:
        """Return all incoming edges (facts) for the given object entity."""
        results = []
        g = self._graph()
        for node in g.nodes():
            if node.lower() == obj.lower():
                for subj, _, data in g.in_edges(node, data=True):
                    results.append({'subject': subj, **data, 'object': node})
        return results

    def by_relation(self, relation: str) -> List[Dict[str, Any]]:
        """Return all triples with a given relation."""
        rel_norm = relation.lower().replace(' ', '_')
        results  = []
        for u, v, data in self._graph().edges(data=True):
            if data.get('relation', '').lower() == rel_norm:
                results.append({'subject': u, **data, 'object': v})
        return results

    def free_text(self, query: str) -> Dict[str, Any]:
        """
        Accept a natural-language query and dispatch to the best
        structured query method. Returns results plus detected intent.
        """
        q = query.strip()
        q_lower = q.lower()

        # Detect pattern: "what do you know about X" / "tell me about X"
        m = re.search(r'(?:about|tell me about|what is|who is)\s+(.+)', q_lower)
        if m:
            entity = m.group(1).strip().title()
            results = self.by_subject(entity) or self.by_object(entity)
            return {'intent': 'entity_lookup', 'entity': entity, 'results': results}

        # Detect relation query: "who born_in Germany" / "what relation born_in"
        m = re.search(r'(?:who|what)\s+(\w+)\s+(\w+)', q_lower)
        if m:
            rel = m.group(2)
            results = self.by_relation(rel)
            return {'intent': 'relation_query', 'relation': rel, 'results': results}

        # Fallback: search nodes by substring
        hits = []
        for node in self._graph().nodes():
            if q_lower in node.lower():
                hits.extend(self.by_subject(node))
        return {'intent': 'substring_match', 'query': q, 'results': hits}

    def format_results(self, results: List[Dict[str, Any]]) -> str:
        """Format query results as a readable string."""
        if not results:
            return 'No results found.'
        lines = []
        for r in results:
            subj = r.get('subject', '?')
            rel  = r.get('relation', '?')
            obj  = r.get('object', '?')
            conf = r.get('confidence', '–')
            conf_str = f'{conf:.2f}' if isinstance(conf, float) else str(conf)
            lines.append(f'  {subj}  →[{rel}]→  {obj}  (conf: {conf_str})')
        return '\n'.join(lines)


print('✅ Query engine class ready.')

---
## Phase 11 — Evaluation Module

The evaluation module computes quality metrics without requiring a gold-standard dataset. It measures extraction yield, confidence distribution, filtering impact, symbolic rule compliance, and graph connectivity. Running the pipeline multiple times and comparing results also tests output stability.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Phase 11 – Evaluation Module
# ─────────────────────────────────────────────────────────────

class EvaluationModule:
    """
    Computes quality metrics for the extraction pipeline.
    Does not require ground-truth annotations.
    """

    @staticmethod
    def compute(run_results: Dict[str, Any]) -> Dict[str, Any]:
        """
        Compute metrics from the output of a pipeline run.
        `run_results` is the dict returned by `process_text()`.
        """
        raw      = run_results.get('raw_triples', [])
        accepted = run_results.get('accepted_triples', [])
        rejected = run_results.get('rejected_triples', [])
        refined  = run_results.get('refined_triples', [])
        inferred = run_results.get('inferred_triples', [])
        graph    = run_results.get('graph_stats', {})

        n_raw      = len(raw)
        n_accepted = len(accepted)
        n_rejected = len(rejected)
        n_refined  = len(refined)
        n_inferred = len(inferred)

        acceptance_rate = (n_accepted / n_raw * 100) if n_raw else 0
        refinement_rate = (n_refined  / n_accepted * 100) if n_accepted else 0
        expansion_rate  = (n_inferred / n_refined  * 100) if n_refined  else 0

        confidences = [t.get('confidence', 0) for t in accepted]
        avg_conf = np.mean(confidences) if confidences else 0
        min_conf = np.min(confidences)  if confidences else 0
        max_conf = np.max(confidences)  if confidences else 0

        return {
            'raw_triples'      : n_raw,
            'accepted_triples' : n_accepted,
            'rejected_triples' : n_rejected,
            'refined_triples'  : n_refined,
            'inferred_triples' : n_inferred,
            'acceptance_rate_pct': round(acceptance_rate, 1),
            'refinement_rate_pct': round(refinement_rate, 1),
            'expansion_rate_pct' : round(expansion_rate, 1),
            'avg_confidence'   : round(float(avg_conf), 3),
            'min_confidence'   : round(float(min_conf), 3),
            'max_confidence'   : round(float(max_conf), 3),
            'graph_nodes'      : graph.get('nodes', 0),
            'graph_edges'      : graph.get('edges', 0),
        }

    @staticmethod
    def stability_check(
        results_a: Dict[str, Any],
        results_b: Dict[str, Any],
    ) -> Dict[str, Any]:
        """
        Compare two evaluation results to assess pipeline stability.
        Returns Jaccard similarity of the accepted triple sets.
        """
        def triple_set(r):
            return {
                (t['subject'], t['relation'], t['object'])
                for t in r.get('refined_triples', [])
            }

        set_a = triple_set(results_a)
        set_b = triple_set(results_b)
        intersection = set_a & set_b
        union        = set_a | set_b
        jaccard = len(intersection) / len(union) if union else 1.0
        return {
            'run_a_triples'  : len(set_a),
            'run_b_triples'  : len(set_b),
            'common_triples' : len(intersection),
            'jaccard_similarity': round(jaccard, 3),
        }


_evaluator = EvaluationModule()
print('✅ Evaluation module ready.')

---
## Phase 12 — Feedback Loop

The feedback loop allows the system to iteratively improve. It identifies common error patterns in rejected triples, suggests prompt refinements, and optionally rewrites symbolic rules. After each iteration the pipeline is re-run so the user can observe improvement in quality metrics.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Phase 12 – Feedback Loop
# ─────────────────────────────────────────────────────────────

class FeedbackLoop:
    """
    Analyses pipeline results to identify failure modes and
    suggest incremental improvements to the prompt and rules.
    """

    def analyse(self, run_results: Dict[str, Any]) -> Dict[str, Any]:
        rejected = run_results.get('rejected_triples', [])

        # Frequency count of rejected relation labels
        rel_counts: Dict[str, int] = defaultdict(int)
        for t in rejected:
            rel_counts[t.get('relation', 'unknown')] += 1

        top_rejected_relations = sorted(rel_counts.items(),
                                        key=lambda x: x[1], reverse=True)[:5]

        suggestions = []
        if run_results.get('raw_triples') == []:
            suggestions.append('LLM returned no triples — consider making the prompt stricter '
                                'or checking Ollama connectivity.')
        if len(rejected) > len(run_results.get('accepted_triples', [])):
            suggestions.append('More triples rejected than accepted — '
                                'consider lowering CONFIDENCE_THRESHOLD or reviewing banned relations.')
        if top_rejected_relations:
            common_rels = [r for r, _ in top_rejected_relations]
            suggestions.append(f'Frequently rejected relations: {common_rels}. '
                                'Consider removing them from the banned list if they are informative.')

        return {
            'top_rejected_relations': top_rejected_relations,
            'suggestions'           : suggestions,
        }

    @staticmethod
    def update_banned_relations(reasoner: SymbolicReasoner,
                                 relations_to_allow: List[str]) -> None:
        """Remove false-positive banned relations based on feedback."""
        for rel in relations_to_allow:
            reasoner.banned_relations.discard(rel)
        logger.info(f'Updated banned relations. Removed: {relations_to_allow}')


_feedback_loop = FeedbackLoop()
print('✅ Feedback loop ready.')

---
## Phase 13 — Backend API (Frontend Integration Layer)

This section assembles the complete pipeline and exposes four clean, callable functions that the Streamlit frontend uses directly. No HTTP server is needed — the frontend imports this notebook's kernel (or the functions are placed in a shared Python module that both the notebook and Streamlit import).

In [ ]:
# ─────────────────────────────────────────────────────────────
# Phase 13 – Backend API
# ─────────────────────────────────────────────────────────────

# ── Module singletons ─────────────────────────────────────────
_preprocessor      = Preprocessor()
_confidence_est    = ConfidenceEstimator()
_sym_reasoner      = SymbolicReasoner()
_kg_builder        = KnowledgeGraphBuilder()
_inference_eng     = InferenceEngine()
_evaluator         = EvaluationModule()
_feedback          = FeedbackLoop()

# ── Global state holding the last pipeline results ────────────
_last_results: Dict[str, Any] = {}


def process_text(input_text: str,
                 accumulate: bool = False) -> Dict[str, Any]:
    """
    Full pipeline: text → knowledge graph.

    Parameters
    ----------
    input_text : raw text to process
    accumulate : if True, new triples are ADDED to the existing graph;
                 if False (default), the graph is rebuilt from scratch.

    Returns
    -------
    Dict with keys: raw_triples, accepted_triples, rejected_triples,
                    refined_triples, inferred_triples, graph_stats,
                    evaluation, feedback.
    """
    global _last_results

    if not accumulate:
        _kg_builder.clear()

    # Phase 2: Preprocess
    pp = Preprocessor.process(input_text)

    # Phase 3: Extract triples (LLM or fallback)
    raw_lines: List[str] = []
    for sentence in pp['sentences']:
        if OLLAMA_AVAILABLE:
            raw_lines.extend(_neural_extractor.extract(sentence))
        else:
            raw_lines.extend(_rule_based_extractor.extract(sentence))

    # Phase 4: Parse
    raw_triples = TripletParser.parse_all(raw_lines)

    # Phase 5: Confidence filtering
    filtered   = _confidence_est.filter_triples(raw_triples)
    accepted   = filtered['accepted']
    rejected   = filtered['rejected']

    # Phase 6: Symbolic reasoning
    refined = _sym_reasoner.apply(accepted)

    # Phase 7 + 8: Build and save graph
    _kg_builder.add_triples(refined)
    StorageManager.save_graph(_kg_builder)
    StorageManager.save_triples(refined, tag='refined')

    # Phase 9: Inference
    inferred = _inference_eng.run(_kg_builder)

    results = {
        'preprocessed'    : pp,
        'raw_triples'     : raw_triples,
        'accepted_triples': accepted,
        'rejected_triples': rejected,
        'refined_triples' : refined,
        'inferred_triples': inferred,
        'graph_stats'     : _kg_builder.get_stats(),
    }

    # Phase 11: Evaluate
    results['evaluation'] = _evaluator.compute(results)

    # Phase 12: Feedback
    results['feedback'] = _feedback.analyse(results)

    _last_results = results
    return results


def get_triples() -> Dict[str, List]:
    """Return the triple sets from the last pipeline run."""
    return {
        'raw'     : _last_results.get('raw_triples', []),
        'accepted': _last_results.get('accepted_triples', []),
        'rejected': _last_results.get('rejected_triples', []),
        'refined' : _last_results.get('refined_triples', []),
        'inferred': _last_results.get('inferred_triples', []),
    }


def get_graph() -> Dict[str, Any]:
    """Return a JSON-serialisable representation of the current knowledge graph."""
    return {
        'graph'     : _kg_builder.to_serialisable(),
        'stats'     : _kg_builder.get_stats(),
        'evaluation': _last_results.get('evaluation', {}),
    }


def query_graph(query: str) -> Dict[str, Any]:
    """
    Query the knowledge graph.
    Supports free-text queries; dispatches to the most appropriate query type.
    """
    engine = QueryEngine(_kg_builder)
    result = engine.free_text(query)
    result['formatted'] = engine.format_results(result.get('results', []))
    return result


print('✅ Backend API ready.')
print('   Functions: process_text(), get_triples(), get_graph(), query_graph()')

---
## Phase 13 — End-to-End Demo Run

Let's run the full pipeline on a sample paragraph and inspect all intermediate outputs.

In [ ]:
# ─────────────────────────────────────────────────────────────
# End-to-End Demo
# ─────────────────────────────────────────────────────────────

DEMO_TEXT = """
Albert Einstein was born in Germany in 1879. He developed the theory of relativity
and worked at Princeton University. Einstein won the Nobel Prize in Physics in 1921.
Marie Curie was born in Warsaw, Poland. She discovered radium and polonium.
Marie Curie won the Nobel Prize in Chemistry. She studied at the University of Paris,
which is located in France. Tim Berners-Lee invented the World Wide Web.
He worked at CERN, which is located in Switzerland.
"""

print('Running full pipeline…')
results = process_text(DEMO_TEXT)

print(f"\n{'='*60}")
print('📋 PIPELINE RESULTS')
print(f"{'='*60}")
print(f"Sentences segmented  : {results['preprocessed']['sent_count']}")
print(f"Raw triples          : {len(results['raw_triples'])}")
print(f"Accepted (confidence): {len(results['accepted_triples'])}")
print(f"Rejected (confidence): {len(results['rejected_triples'])}")
print(f"After symbolic rules : {len(results['refined_triples'])}")
print(f"Inferred (multi-hop) : {len(results['inferred_triples'])}")
print(f"Graph nodes          : {results['graph_stats']['nodes']}")
print(f"Graph edges          : {results['graph_stats']['edges']}")

print(f"\n{'─'*60}")
print('🧩 REFINED TRIPLES')
for t in results['refined_triples']:
    print(f"  ({t['subject']}, {t['relation']}, {t['object']})  conf={t.get('confidence','–')}")

if results['inferred_triples']:
    print(f"\n{'─'*60}")
    print('🔁 INFERRED TRIPLES')
    for t in results['inferred_triples']:
        print(f"  ({t['subject']}, {t['relation']}, {t['object']})  via: {t.get('via','')}")

print(f"\n{'─'*60}")
print('📊 EVALUATION')
for k, v in results['evaluation'].items():
    print(f"  {k:28s}: {v}")

print(f"\n{'─'*60}")
print('💡 FEEDBACK')
for s in results['feedback']['suggestions']:
    print(f"  • {s}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Query Demo
# ─────────────────────────────────────────────────────────────

queries = [
    'tell me about Albert Einstein',
    'what do you know about Marie Curie',
    'who born_in Germany',
    'Tim Berners-Lee',
]

for q in queries:
    print(f"\n🔍 Query: '{q}'")
    result = query_graph(q)
    print(result['formatted'] or 'No results.')

---
## Phase 14 — Documentation

### System Architecture

The system is built as a 14-phase pipeline inside a single notebook. Every phase is implemented as a self-contained Python class with clear input and output contracts. The phases are:

1. **Initialisation** — library imports, Ollama connectivity check.
2. **Preprocessing** — noise removal, normalisation, sentence segmentation.
3. **Neural Extraction** — LLM prompt engineering, retry logic, rule-based fallback.
4. **Triplet Parsing** — regex-based parsing, normalisation, deduplication.
5. **Confidence Estimation** — heuristic scoring, threshold filtering, logging.
6. **Symbolic Reasoning** — rule enforcement, entity canonicalisation, deduplication.
7. **Graph Construction** — NetworkX MultiDiGraph with edge attributes.
8. **Storage** — JSON persistence to disk, session reload.
9. **Inference Engine** — transitive rule application, 2-hop+ reasoning.
10. **Query Module** — subject/object/relation queries, free-text dispatch.
11. **Evaluation** — unsupervised quality metrics, stability checks.
12. **Feedback Loop** — error pattern analysis, prompt/rule suggestions.
13. **Backend API** — four public functions consumed by the Streamlit UI.
14. **Documentation** — this section.

### Frontend–Backend Integration

The Streamlit frontend (`app.py`) imports the backend functions by placing both files in the same directory and using Python's standard import mechanism. The data flow is:

```
User types text → app.py calls process_text() → pipeline runs → results returned
User submits query → app.py calls query_graph() → QueryEngine traverses graph → answer displayed
```

No intermediate HTTP layer is needed.

### Limitations

- LLM extraction quality depends on model capability; LLaMA3-8B may miss subtle relations.
- The rule-based fallback covers common English patterns only; domain-specific texts may need additional patterns.
- Confidence scoring uses structural heuristics, not a trained classifier.
- The inference engine applies forward-chaining rules; backward-chaining and complex ontological reasoning are beyond current scope.

### Conclusion

This system demonstrates a practical, fully integrated Neural–Symbolic Knowledge Base Construction pipeline that operates end-to-end from raw text to a queryable graph. It is designed for extensibility: any module can be replaced or upgraded independently. The Streamlit frontend provides a clean, real-time interface for non-technical users to interact with the knowledge graph.